In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("C:/Users/axere/OneDrive/Desktop/DA projects/Data sets/Flats uncleaned data/archive/surat_uncleaned.csv")

def Cleaned_data(df):
    final_col = ['property_name', 'area_sqft', 'area_type', 
                  'transactions', 'status', 'floor_num', 'total_floors', 
                  'furnishing', 'facing', 'price_val', 'price_per_sqft_val']
    clean_df = pd.DataFrame(index=df.index, columns=final_col)
    clean_df["property_name"] = df["property_name"]

# To extract price
    def extract_price(x):
        if pd.isna(x) or "Call for Price" in str(x):
            return np.nan
        x= str(x).replace("₹","").replace(",","").strip()
        if "Lac" in x :
            return float(x.replace("Lac","").strip())*1e5
        if "Cr" in x:
            return float(x.replace("Cr","").strip())*1e7
        try: 
            return float(x)
        except:
            return np.nan
    clean_df["price_val"] = df["price"].apply(extract_price)

# To extract price per sqft
    def extract_per_sqft(x):
        if pd.isna(x): 
            return np.nan
        match = re.search(r'₹([\d,]+)', str(x))
        if match:
            return float(match.group(1).replace(",",""))
        return np.nan
    clean_df['price_per_sqft_val'] = df["price_per_sqft"].apply(extract_per_sqft)

# Clean Area
    def extract_area(x):
        if pd.isna(x):
            return np.nan
        match = re.search(r'([\d,]+)',str(x))
        if match:
            return float (match.group(1).replace(",",""))
        return np.nan
    clean_df["area_sqft"] = df["square_feet"].apply(extract_area)   

# Getting Location from property name

    def get_loc(x):
        match = re.search(r'in (.*?) Surat', str(x), re.IGNORECASE)
        return match.group(1).split(',')[-1].strip() if match else np.nan
    clean_df['property_location'] = df['property_name'].apply(get_loc)
    
# Cleaning jumbled mess
    def fix_jumble(x):
        row_val = [str(val).strip() for val in x.values if pd.notna(val)]
        res = {'area_type': np.nan, 'transactions': np.nan, 'status': np.nan, 
               'furnishing': np.nan, 'facing': np.nan, 'floor_num': np.nan, 'total_floors': np.nan}
        for v in row_val:
            v1 = v.lower()
            if 'area' in v1:
                if 'carpet' in v1: res["area_type"] = "Carpet Area"
                elif 'super' in v1: res["area_type"] = "Super Area"
                elif 'plot' in v1: res["area_type"] = "Plot Area"
                elif 'built' in v1: res["area_type"] = "Built up Area"
            elif 'new property' in v1: res['transactions'] = 'New Property'
            elif 'resale' in v1: res['transactions'] = 'Resale'
            elif 'ready to move' in v1: res['status'] = 'Ready to Move'
            elif 'poss.' in v1: res['status'] = v
            elif 'unfurnished' in v1: res['furnishing'] = 'Unfurnished'
            elif 'semi-furnished' in v1: res['furnishing'] = 'Semi-Furnished'
            elif 'furnished' in v1: res['furnishing'] = 'Furnished'
            elif any( d in v1 for d in ['north','south','east', 'west']) and len(v)<15: res["facing"] = v
            elif 'out of' in v1:
                part = v1.split("out of")
                try:
                    res["floor_num"] = int(re.search(r'\d+', part[0]).group())
                    res["total_floors"] = int(re.search(r'\d+', part[1]).group())
                except:
                    pass
        return pd.Series(res)
    jumble_extracted = df.apply(fix_jumble, axis= 1)
    for col in jumble_extracted.columns:
        clean_df[col]=jumble_extracted[col]
    return clean_df

df_final = Cleaned_data(df)

#Dealing with missing values
fill_col = ["area_type","transactions","status","furnishing","facing"]
df_final[fill_col] = df_final[fill_col].fillna("unknown")

price_median = df_final["price_val"].median()
df_final["price_val"] = df_final["price_val"].fillna(price_median) 

df_final["price_per_sqft_val"] = df_final["price_per_sqft_val"].fillna(df_final["price_val"] / df_final["area_sqft"])

flooring = ["floor_num","total_floors"]
df_final[flooring] = df_final[flooring].fillna(-1)
df_final=df_final.dropna(subset=["area_sqft"])
df_final["property_location"]= df_final["property_location"].fillna("Unknown")

#Final clean csv
df_final.to_csv('surat_cleaned_01.csv', index=False)


In [2]:
df_final.isnull().sum()

property_name         0
area_sqft             0
area_type             0
transactions          0
status                0
floor_num             0
total_floors          0
furnishing            0
facing                0
price_val             0
price_per_sqft_val    0
property_location     0
dtype: int64